# Lesson 7.1: Color Models and Conversions
## Biomedical Image Processing - Color Image Processing

### Topics:
- RGB color model recap
- CMY and CMYK color models
- HSI (Hue, Saturation, Intensity) color model
- Converting between color models

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("Libraries imported successfully!")

## 1. RGB Color Model - Recap

The **RGB** model is an **additive** color model. Colors are created by combining Red, Green, and Blue light.

- Each pixel has 3 channels: **R**, **G**, **B**
- Each channel has values in the range **[0, 255]** (8-bit)
- Total number of colors: $256^3 = 16{,}777{,}216 \approx 16.7$ million

The RGB model is used by displays, cameras, and most digital imaging hardware.

In [ ]:
# Create a 200x200 test image with colored regions
img = np.zeros((200, 200, 3), dtype=np.uint8)

# Top row: Red, Green, Blue
img[0:100, 0:67, :]   = [255, 0, 0]     # Red
img[0:100, 67:134, :] = [0, 255, 0]     # Green
img[0:100, 134:200, :] = [0, 0, 255]    # Blue

# Bottom row: Yellow, Cyan, Magenta
img[100:200, 0:67, :]   = [255, 255, 0]   # Yellow
img[100:200, 67:134, :] = [0, 255, 255]   # Cyan
img[100:200, 134:200, :] = [255, 0, 255]  # Magenta

# Display original and individual channels
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(img)
axes[0].set_title("Original RGB")

# Show each channel as a grayscale image
channel_names = ['Red Channel', 'Green Channel', 'Blue Channel']
for i in range(3):
    axes[i+1].imshow(img[:, :, i], cmap='gray', vmin=0, vmax=255)
    axes[i+1].set_title(channel_names[i])

for ax in axes:
    ax.axis('off')

plt.tight_layout()
plt.show()

print("Notice how each channel is bright (255) only where that color is present.")

## 2. CMY and CMYK Color Models

The **CMY** model is a **subtractive** color model, primarily used in **printing**.

Instead of adding light (like RGB), CMY works by subtracting colors from white light using inks:
- **C**yan absorbs Red
- **M**agenta absorbs Green
- **Y**ellow absorbs Blue

### RGB to CMY Conversion (normalized to [0, 1]):

$$C = 1 - R, \quad M = 1 - G, \quad Y = 1 - B$$

### CMYK adds a Key (Black) channel for printing efficiency:

$$K = \min(C, M, Y)$$
$$C' = C - K, \quad M' = M - K, \quad Y' = Y - K$$

In [ ]:
# RGB to CMY conversion
# Normalize RGB to [0, 1]
img_norm = img.astype(np.float64) / 255.0

# CMY = 1 - RGB
C = 1.0 - img_norm[:, :, 0]
M = 1.0 - img_norm[:, :, 1]
Y = 1.0 - img_norm[:, :, 2]

# CMYK: K = min(C, M, Y)
K = np.minimum(np.minimum(C, M), Y)
C_k = C - K
M_k = M - K
Y_k = Y - K

# Visualize RGB channels vs CMY channels
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

# Top row: RGB channels
axes[0, 0].imshow(img_norm[:, :, 0], cmap='Reds', vmin=0, vmax=1)
axes[0, 0].set_title("R Channel")
axes[0, 1].imshow(img_norm[:, :, 1], cmap='Greens', vmin=0, vmax=1)
axes[0, 1].set_title("G Channel")
axes[0, 2].imshow(img_norm[:, :, 2], cmap='Blues', vmin=0, vmax=1)
axes[0, 2].set_title("B Channel")

# Bottom row: CMY channels
axes[1, 0].imshow(C, cmap='gray', vmin=0, vmax=1)
axes[1, 0].set_title("C (Cyan) Channel")
axes[1, 1].imshow(M, cmap='gray', vmin=0, vmax=1)
axes[1, 1].set_title("M (Magenta) Channel")
axes[1, 2].imshow(Y, cmap='gray', vmin=0, vmax=1)
axes[1, 2].set_title("Y (Yellow) Channel")

for ax in axes.flat:
    ax.axis('off')

plt.suptitle("RGB vs CMY Channels", fontsize=14)
plt.tight_layout()
plt.show()

# Show CMYK K channel
fig, ax = plt.subplots(1, 1, figsize=(4, 4))
ax.imshow(K, cmap='gray', vmin=0, vmax=1)
ax.set_title("K (Black) Channel")
ax.axis('off')
plt.show()

print("K channel is non-zero only where all three CMY values are > 0.")
print("In our test image, K = 0 everywhere (pure colors have at least one RGB channel at max).")

## 3. HSI Color Model

The **HSI** (Hue, Saturation, Intensity) model separates color information from brightness, which is closer to how humans perceive color.

- **Hue (H):** The angle on the color circle ($0°$-$360°$). Represents the dominant wavelength.
- **Saturation (S):** Purity of the color ($0$-$1$). Low saturation = washed out / grayish.
- **Intensity (I):** Average brightness ($0$-$1$). Independent of color.

### RGB to HSI Formulas:

$$I = \frac{R + G + B}{3}$$

$$S = 1 - \frac{3 \cdot \min(R, G, B)}{R + G + B}$$

$$\theta = \cos^{-1}\left(\frac{0.5\,[(R-G) + (R-B)]}{\sqrt{(R-G)^2 + (R-B)(G-B)}}\right)$$

$$H = \begin{cases} \theta & \text{if } B \leq G \\ 360° - \theta & \text{if } B > G \end{cases}$$

> **Note:** When $R = G = B$ (grayscale), hue is undefined and saturation is 0.

In [ ]:
def rgb_to_hsi(img):
    """
    Convert an RGB image (uint8) to HSI.
    Returns: H (0-360 degrees), S (0-1), I (0-1)
    """
    # Normalize to [0, 1]
    rgb = img.astype(np.float64) / 255.0
    R = rgb[:, :, 0]
    G = rgb[:, :, 1]
    B = rgb[:, :, 2]
    
    # Intensity
    I = (R + G + B) / 3.0
    
    # Saturation
    min_rgb = np.minimum(np.minimum(R, G), B)
    sum_rgb = R + G + B
    # Avoid division by zero for black pixels
    S = np.where(sum_rgb == 0, 0, 1.0 - 3.0 * min_rgb / sum_rgb)
    
    # Hue
    numerator = 0.5 * ((R - G) + (R - B))
    denominator = np.sqrt((R - G)**2 + (R - B) * (G - B))
    # Avoid division by zero
    denominator = np.where(denominator == 0, 1e-10, denominator)
    
    theta = np.arccos(np.clip(numerator / denominator, -1.0, 1.0))
    theta = np.degrees(theta)  # Convert to degrees
    
    H = np.where(B <= G, theta, 360.0 - theta)
    
    # When S=0 (grayscale), hue is undefined -> set to 0
    H = np.where(S == 0, 0, H)
    
    return H, S, I

print("rgb_to_hsi() function defined successfully!")

In [ ]:
# Apply HSI conversion to the test image
H, S, I = rgb_to_hsi(img)

# Visualize the HSI channels
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(img)
axes[0].set_title("Original RGB")

axes[1].imshow(H, cmap='hsv', vmin=0, vmax=360)
axes[1].set_title("Hue (0°-360°)")

axes[2].imshow(S, cmap='gray', vmin=0, vmax=1)
axes[2].set_title("Saturation (0-1)")

axes[3].imshow(I, cmap='gray', vmin=0, vmax=1)
axes[3].set_title("Intensity (0-1)")

for ax in axes:
    ax.axis('off')

plt.tight_layout()
plt.show()

# Print HSI values for each color region
colors = ['Red', 'Green', 'Blue', 'Yellow', 'Cyan', 'Magenta']
positions = [(50, 33), (50, 100), (50, 167), (150, 33), (150, 100), (150, 167)]

print("\nHSI values for each color region:")
print(f"{'Color':<10} {'H (deg)':>8} {'S':>6} {'I':>6}")
print("-" * 32)
for name, (r, c) in zip(colors, positions):
    print(f"{name:<10} {H[r,c]:>8.1f} {S[r,c]:>6.2f} {I[r,c]:>6.2f}")

## Summary

What we learned:
1. **RGB model** = additive color model with 3 channels (R, G, B), each in [0, 255]
2. **CMY/CMYK model** = subtractive model for printing; $C = 1-R$, $M = 1-G$, $Y = 1-B$
3. **HSI model** = separates color (Hue), purity (Saturation), and brightness (Intensity)
4. **RGB to HSI conversion** = Intensity is the average, Saturation measures color purity, Hue is the color angle
5. **HSI advantage** = processing intensity independently preserves color information